plan:
generate noisy circle
compute VR (Ripser) and Alpha(GUDHI)
compute Delaunay Triangulation (scipy.spatial) to see underlying Alpha complex
use persim for diagram comparison (Wasserstein/Bottleneck distance)
compare persistence diagrams
create comparison table
draw conclusions

In [ ]:
import os
import tadasets
import gudhi
import numpy as np
import matplotlib.pyplot as plt
import time
from ripser import ripser
from persim import plot_diagrams


# ── Data helpers ──────────────────────────────────────────────────────────────

def compute_comparison(shape, n_points, noise, max_dim=2):
    """Generate a point cloud and compute Rips + Alpha persistence."""
    if shape == 'circle':
        pts = tadasets.dsphere(n=n_points, d=1, r=1, noise=noise)
    elif shape == 'torus':
        pts = tadasets.torus(n=n_points, c=2, a=1, noise=noise)
    elif shape == 'sphere':
        pts = tadasets.dsphere(n=n_points, d=2, r=1, noise=noise)
    else:
        raise ValueError(f"Unknown shape: {shape!r}")

    t0 = time.time()
    result    = ripser(pts, maxdim=max_dim)
    rips_ms   = (time.time() - t0) * 1000
    dgms_rips = result['dgms']
    num_edges = result['num_edges']

    t0 = time.time()
    ac  = gudhi.AlphaComplex(points=pts)
    st  = ac.create_simplex_tree()
    st.compute_persistence()
    alpha_ms      = (time.time() - t0) * 1000
    num_simplices = st.num_simplices()
    dgms_alpha    = [
        np.array([[b, d] for dim, (b, d) in st.persistence()
                  if dim == i and d != float('inf')]
                 or np.empty((0, 2)))
        for i in range(max_dim + 1)
    ]

    return pts, dgms_rips, num_edges, rips_ms, dgms_alpha, num_simplices, alpha_ms


def _filter_dgms(dgms, threshold):
    """Keep infinite bars always; drop finite bars with persistence ≤ threshold."""
    if threshold <= 0:
        return dgms
    out = []
    for dgm in dgms:
        if len(dgm) == 0:
            out.append(dgm)
        else:
            inf_mask = ~np.isfinite(dgm[:, 1])
            fin_mask = np.isfinite(dgm[:, 1]) & ((dgm[:, 1] - dgm[:, 0]) > threshold)
            out.append(dgm[inf_mask | fin_mask])
    return out


# ── Plot helpers ──────────────────────────────────────────────────────────────

def plot_barcode(dgms, ax, title="Barcode"):
    colors = ['tab:blue', 'tab:orange', 'tab:green']
    y = 0
    for dim, dgm in enumerate(dgms):
        if len(dgm) == 0:
            continue
        for b, d in dgm:
            if np.isfinite(d):
                ax.plot([b, d], [y, y], color=colors[dim % len(colors)],
                        linewidth=1.5, solid_capstyle='butt')
                y += 1
    ax.set_title(title)
    ax.set_yticks([])
    ax.set_xlabel("Filtration value")


def plot_landscape(dgms, ax, title="Landscape", n_landscapes=3):
    colors = ['tab:blue', 'tab:orange', 'tab:green']
    plotted = False
    for dim, dgm in enumerate(dgms):
        if len(dgm) == 0:
            continue
        finite = dgm[np.isfinite(dgm[:, 1])]
        if len(finite) == 0:
            continue
        t_min, t_max = finite[:, 0].min(), finite[:, 1].max()
        ts    = np.linspace(t_min, t_max, 500)
        tents = np.maximum(0, np.minimum(ts[None, :] - finite[:, 0:1],
                                         finite[:, 1:2] - ts[None, :]))
        sorted_tents = np.sort(tents, axis=0)[::-1]
        for k in range(min(n_landscapes, len(sorted_tents))):
            ax.plot(ts, sorted_tents[k], color=colors[dim % len(colors)],
                    alpha=max(0.3, 0.9 - 0.25 * k), linewidth=1,
                    label=f'H{dim} λ{k+1}')
        plotted = True
    ax.set_title(title)
    ax.set_xlabel("Filtration value")
    if plotted:
        ax.legend(fontsize=6)


# ── Primary comparison function ───────────────────────────────────────────────

def plot_comparison(pts,
                    dgms_rips, num_edges, rips_ms,
                    dgms_alpha, num_simplices, alpha_ms,
                    shape, n_points, noise, threshold=0.0,
                    show_pd=True, show_bc=True, show_pl=True,
                    save=False):
    """
    Render the 2-row Rips / Alpha comparison for one config.

    Columns shown are controlled by show_pd, show_bc, show_pl.
    If all three are False, all three columns are shown (default-all convention).
    Set save=True to write a PNG to tda_outputs/.
    """
    dgms_rips_f  = _filter_dgms(dgms_rips,  threshold)
    dgms_alpha_f = _filter_dgms(dgms_alpha, threshold)

    col_types = [t for flag, t in [(show_pd, 'pd'), (show_bc, 'bc'), (show_pl, 'pl')] if flag]
    if not col_types:
        col_types = ['pd', 'bc', 'pl']

    n_diag = len(col_types)
    fig = plt.figure(figsize=(6 + 5 * n_diag, 9), layout='constrained')
    fig.suptitle(
        f"{shape.capitalize()}  |  n={n_points}  |  noise={noise}  |  thresh={threshold}",
        fontsize=12, fontweight='bold'
    )
    gs = fig.add_gridspec(2, 1 + n_diag)

    # Column 0: point cloud spanning both rows
    if pts.shape[1] == 2:
        ax_pc = fig.add_subplot(gs[:, 0])
        ax_pc.scatter(pts[:, 0], pts[:, 1], s=5)
        ax_pc.set_aspect('equal')
    else:
        ax_pc = fig.add_subplot(gs[:, 0], projection='3d')
        ax_pc.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2)
    ax_pc.set_title("Point Cloud")

    # Diagram columns
    for col_offset, col_type in enumerate(col_types):
        c = 1 + col_offset

        if col_type == 'pd':
            ax_r = fig.add_subplot(gs[0, c])
            plot_diagrams(dgms_rips_f, show=False, ax=ax_r)
            ax_r.set_title(f"Rips Diagram\nedges: {num_edges}  |  {rips_ms:.1f} ms")

            ax_a = fig.add_subplot(gs[1, c])
            plot_diagrams(dgms_alpha_f, show=False, ax=ax_a)
            ax_a.set_title(f"Alpha Diagram\nsimplices: {num_simplices}  |  {alpha_ms:.1f} ms")

        elif col_type == 'bc':
            ax_r = fig.add_subplot(gs[0, c])
            plot_barcode(dgms_rips_f, ax_r, title="Rips Barcode")

            ax_a = fig.add_subplot(gs[1, c])
            plot_barcode(dgms_alpha_f, ax_a, title="Alpha Barcode")

        elif col_type == 'pl':
            ax_r = fig.add_subplot(gs[0, c])
            plot_landscape(dgms_rips_f, ax_r, title="Rips Landscape")

            ax_a = fig.add_subplot(gs[1, c])
            plot_landscape(dgms_alpha_f, ax_a, title="Alpha Landscape")

    if save:
        os.makedirs('tda_outputs', exist_ok=True)
        shown    = '+'.join(col_types)
        filename = f'tda_outputs/{shape}_n{n_points}_ns{noise}_{shown}_th{threshold}.png'
        plt.savefig(filename, dpi=150, bbox_inches='tight')

    plt.show()

In [ ]:
# configs: (shape, n_points, noise, threshold)
configs = [
    ('circle', 100, 0.05, 0.0),
    ('circle', 200, 0.10, 0.0),
    ('circle', 200, 0.30, 0.05),
    ('torus',  300, 0.05, 0.0),
    ('torus',  500, 0.10, 0.0),
]

for shape, n_points, noise, threshold in configs:
    pts, dgms_rips, num_edges, rips_ms, dgms_alpha, num_simplices, alpha_ms = \
        compute_comparison(shape, n_points, noise)
    plot_comparison(
        pts, dgms_rips, num_edges, rips_ms, dgms_alpha, num_simplices, alpha_ms,
        shape=shape, n_points=n_points, noise=noise, threshold=threshold,
        show_pd=True, show_bc=True, show_pl=True, save=True
    )